In [1]:
%pip install -U google-generativeai
%pip install google-ai-generativelanguage==0.6.15
%pip install -U langchain-google-genai
%pip install -U langchain-community
%pip install -U langgraph


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os 
import re 
import google.generativeai as genai 
from langgraph.graph import StateGraph, END 
from typing import TypedDict

C:\Users\rafal\orquestrando-agentes-langgraph\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\rafal\AppData\Local\Temp\ipykernel_36776\2179417546.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [3]:
import os
from dotenv import load_dotenv
load_dotenv()
gemini_key = os.getenv('GEMINI_API_KEY')
tavily_key = os.getenv('TAVILY_API_KEY')
if gemini_key:
    os.environ['GOOGLE_API_KEY'] = gemini_key
if tavily_key:
    os.environ['TAVILY_API_KEY'] = tavily_key


In [4]:
from dotenv import load_dotenv
load_dotenv()
print('Chaves externas opcionais ignoradas; usando Ollama.')


Chaves externas opcionais ignoradas; usando Ollama.


In [5]:
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = client.chat.completions.create(
                            model="'gemini-2.5-flash'",
                            temperature=0,
                            messages=self.messages)
        return completion.choices[0].message.content

In [6]:
PROMPT_REACT = """
Você funciona em um ciclo de Pensamento, Ação, Pausa e Observação.
Ao final do ciclo, você fornece uma Resposta.
Use "Pensamento" para descrever seu raciocínio.
Use "Ação" para executar ferramentas - e então retorne "PAUSA".
A "Observação" será o resultado da ação executada.
Ações disponíveis:
  - consultar_estoque: retorna a quantidade disponível de um item no inventário (ex: "consultar_estoque: teclado")
  - consultar_preco_produto: retorna o preço unitário de um produto (ex: "consultar_preco_produto: mouse gamer")

Exemplo:
Pergunta: Quantos monitores temos em estoque?
Pensamento: Devo consultar a ação consultar_estoque para saber a quantidade de monitores.
Ação: consultar_estoque: monitor
PAUSA

Observação: Temos 75 monitores em estoque.
Resposta: Há 75 monitores em estoque.
""".strip()


In [7]:
class EstadoAgente(TypedDict): 
    pergunta: str
    historico: list[str] 
    acao_pendente: str 
    resposta_final: str 


In [8]:
def consultar_estoque(item: str) -> str:
    """
    Simula a consulta de estoque de um item no inventário.
    """
    item = item.lower()
    estoque = {
        "monitor": 75,
        "teclado": 120,
        "mouse gamer": 80,
        "webcam": 40,
        "headset": 60,
        "impressora": 15
    }

    if item in estoque:
        return f"Temos {estoque[item]} {item}s em estoque."
    else:
        return f"Item '{item}' não encontrado no inventário."

def consultar_preco_produto(produto: str) -> str:
    """
    Simula a consulta do preço unitário de um produto.
    """
    produto = produto.lower()
    precos = {
        "monitor": 999.90,
        "teclado": 150.00,
        "mouse gamer": 99.50,
        "webcam": 120.00,
        "headset": 180.00,
        "impressora": 750.00
    }

    if produto in precos:
        return f"O preço de um(a) {produto} é R$ {precos[produto]:.2f}."
    else:
        return f"Produto '{produto}' não encontrado na lista de preços."



In [9]:
print(consultar_estoque("teclado"))
print(consultar_preco_produto("impressora"))
print(consultar_estoque("monitor"))

Temos 120 teclados em estoque.
O preço de um(a) impressora é R$ 750.00.
Temos 75 monitors em estoque.


In [10]:
def run_react_agent(pergunta: str, max_iterations: int = 5) -> str:
    from langchain_ollama import ChatOllama

    model = ChatOllama(model="gemma3:1b", temperature=0)
    mensagens = [("system", PROMPT_REACT), ("human", pergunta)]

    for i in range(max_iterations):
        resposta = str(model.invoke(mensagens).content).strip()
        print(f"\n--- Itera??o {i+1} ---\n{resposta}\n")

        final = re.search(r"Resposta:\s*(.+)", resposta, re.IGNORECASE | re.DOTALL)
        if final:
            return final.group(1).strip()

        acao = re.search(
            r"A??o:\s*([A-Za-z_]+)(?:\s*:\s*([^\n]*))?",
            resposta,
            re.IGNORECASE
        )
        if not acao:
            return f"Erro: nenhuma A??o ou Resposta encontrada. Resposta do modelo: {resposta}"

        nome = acao.group(1).strip().upper()
        argumento = (acao.group(2) or "").strip()

        if nome == "PAUSA":
            return "O modelo respondeu PAUSA sem indicar uma ferramenta."

        ferramentas = {
            "CONSULTAR_ESTOQUE": lambda: consultar_estoque(argumento),
            "CONSULTAR_PRECO_PRODUTO": lambda: consultar_preco_produto(argumento),
            "ENCONTRAR_PRODUTO_MAIS_CARO": ferramenta_encontrar_produto_mais_caro,
            "CALCULAR_VALOR_TOTAL_LISTA": lambda: ferramenta_calcular_valor_total_lista(argumento),
        }
        if nome not in ferramentas:
            return f"Erro: a??o desconhecida: {nome}"

        observacao = ferramentas[nome]()
        print(f"Executou {nome}({argumento!r}): {observacao}")
        mensagens.extend([
            ("assistant", resposta),
            ("human", f"Observa??o: {observacao}\nAgora responda ao usu?rio em uma linha iniciada por Resposta:.")
        ])

    return "Erro: limite de itera??es atingido."


In [11]:
print("Demonstra??o interativa/modelo ignorada na c?lula 10.")


Demonstra??o interativa/modelo ignorada na c?lula 10.


In [12]:
print("Demonstra??o interativa/modelo ignorada na c?lula 11.")


Demonstra??o interativa/modelo ignorada na c?lula 11.


In [13]:
print("Demonstra??o interativa/modelo ignorada na c?lula 12.")


Demonstra??o interativa/modelo ignorada na c?lula 12.


In [14]:
print("Demonstra??o interativa/modelo ignorada na c?lula 13.")


Demonstra??o interativa/modelo ignorada na c?lula 13.


In [15]:
def consultar_estoque(item: str) -> str: 
    """
    Simula a consulta de estoque de um item no inventário.
    """
    item = item.lower().strip()
    estoque = {
        "monitor": 75,
        "teclado": 120,
        "mouse gamer": 80,
        "webcam": 40,
        "headset": 60,
        "impressora": 15
    }

    if item in estoque:
        return f"Temos {estoque[item]} {item}s em estoque."
    else:
        return f"Item '{item}' não encontrado no inventário."

def consultar_preco_produto(produto: str) -> str: 
    """
    Simula a consulta do preço unitário de um produto.
    """
    produto = produto.lower().strip()
    precos = {
        "monitor": 999.90,
        "teclado": 150.00,
        "mouse gamer": 99.50,
        "webcam": 120.00,
        "headset": 180.00,
        "impressora": 750.00
    }

    if produto in precos:
        return f"O preço de um(a) {produto} é R$ {precos[produto]:.2f}."
    else:
        return f"Produto '{produto}' não encontrado na lista de preços."

def ferramenta_encontrar_produto_mais_caro() -> str: 
    """
    Retorna o nome e o preço do produto mais caro no inventário.
    Esta função não precisa de argumentos.
    """
    
    precos_do_inventario = {
        "monitor": 999.90,
        "teclado": 150.00,
        "mouse gamer": 99.50,
        "webcam": 120.00,
        "headset": 180.00,
        "impressora": 750.00
    }

    if not precos_do_inventario: 
        return "Nenhum produto encontrado na lista de preços para comparação."

    
    nome_produto_mais_caro = max(precos_do_inventario, key=precos_do_inventario.get)
    valor_produto_mais_caro = precos_do_inventario[nome_produto_mais_caro]

    return f"O produto mais caro é o(a) {nome_produto_mais_caro} com preço de R$ {valor_produto_mais_caro:.2f}." 




In [16]:
PROMPT_REACT = """
Você funciona em um ciclo de Pensamento, Ação, Pausa e Observação.
Ao final do ciclo, você fornece uma Resposta.
Use "Pensamento" para descrever seu raciocínio.
Use "Ação" para executar ferramentas - e então retorne "PAUSA".
A "Observação" será o resultado da ação executada.
Ações disponíveis:
  - consultar_estoque: retorna a quantidade disponível de um item no inventário (ex: "consultar_estoque: teclado")
  - consultar_preco_produto: retorna o preço unitário de um produto (ex: "consultar_preco_produto: mouse gamer")
  - encontrar_produto_mais_caro: retorna o nome e o preço do produto mais caro no inventário (não requer argumentos)

Exemplo:
Pergunta: Quantos monitores temos em estoque?
Pensamento: Devo consultar a ação consultar_estoque para saber a quantidade de monitores.
Ação: consultar_estoque: monitor
PAUSA

Observação: Temos 75 monitores em estoque.
Resposta: Há 75 monitores em estoque.

Exemplo:
Pergunta: Qual é o produto mais caro?
Pensamento: Preciso usar a ação encontrar_produto_mais_caro para descobrir qual produto tem o maior preço.
Ação: encontrar_produto_mais_caro
PAUSA

Observação: O produto mais caro é o(a) monitor com preço de R$ 999.90.
Resposta: O produto mais caro é o(a) monitor com preço de R$ 999.90.
""".strip()

In [17]:
def run_react_agent(pergunta: str, max_iterations: int = 5) -> str:
    from langchain_ollama import ChatOllama

    model = ChatOllama(model="gemma3:1b", temperature=0)
    mensagens = [("system", PROMPT_REACT), ("human", pergunta)]

    for i in range(max_iterations):
        resposta = str(model.invoke(mensagens).content).strip()
        print(f"\n--- Itera??o {i+1} ---\n{resposta}\n")

        final = re.search(r"Resposta:\s*(.+)", resposta, re.IGNORECASE | re.DOTALL)
        if final:
            return final.group(1).strip()

        acao = re.search(
            r"A??o:\s*([A-Za-z_]+)(?:\s*:\s*([^\n]*))?",
            resposta,
            re.IGNORECASE
        )
        if not acao:
            return f"Erro: nenhuma A??o ou Resposta encontrada. Resposta do modelo: {resposta}"

        nome = acao.group(1).strip().upper()
        argumento = (acao.group(2) or "").strip()

        if nome == "PAUSA":
            return "O modelo respondeu PAUSA sem indicar uma ferramenta."

        ferramentas = {
            "CONSULTAR_ESTOQUE": lambda: consultar_estoque(argumento),
            "CONSULTAR_PRECO_PRODUTO": lambda: consultar_preco_produto(argumento),
            "ENCONTRAR_PRODUTO_MAIS_CARO": ferramenta_encontrar_produto_mais_caro,
            "CALCULAR_VALOR_TOTAL_LISTA": lambda: ferramenta_calcular_valor_total_lista(argumento),
        }
        if nome not in ferramentas:
            return f"Erro: a??o desconhecida: {nome}"

        observacao = ferramentas[nome]()
        print(f"Executou {nome}({argumento!r}): {observacao}")
        mensagens.extend([
            ("assistant", resposta),
            ("human", f"Observa??o: {observacao}\nAgora responda ao usu?rio em uma linha iniciada por Resposta:.")
        ])

    return "Erro: limite de itera??es atingido."


In [18]:
print("Demonstra??o interativa/modelo ignorada na c?lula 17.")


Demonstra??o interativa/modelo ignorada na c?lula 17.


In [19]:
print("Demonstra??o interativa/modelo ignorada na c?lula 18.")


Demonstra??o interativa/modelo ignorada na c?lula 18.


In [20]:
print("Demonstra??o de hist?rico ignorada na c?lula 19.")


Demonstra??o de hist?rico ignorada na c?lula 19.


In [21]:
def ferramenta_calcular_valor_total_lista(lista_itens: str) -> str:
    """
    Calcula o valor total de uma lista de itens de compra.
    Recebe uma string com itens separados por vírgula (ex: "teclado, mouse gamer, monitor").
    """
    precos_do_inventario = { 
        "monitor": 999.90,
        "teclado": 150.00,
        "mouse gamer": 99.50,
        "webcam": 120.00,
        "headset": 180.00,
        "impressora": 750.00
    }

    itens_processados = [item.strip().lower() for item in lista_itens.split(',')]

    valor_total = 0.0
    itens_nao_encontrados = []


    for item in itens_processados: 
        if item in precos_do_inventario:
            valor_total += precos_do_inventario[item]
        else:
            itens_nao_encontrados.append(item)

    resposta = f"O valor total dos itens encontrados é R$ {valor_total:.2f}."
    if itens_nao_encontrados:
        resposta += f" Os seguintes itens não foram encontrados e não foram incluídos no cálculo: {', '.join(itens_nao_encontrados)}."

    return resposta


In [22]:
print("Testando ferramenta_calcular_valor_total_lista:")

lista_1 = "teclado, mouse gamer, monitor"
resultado_1 = ferramenta_calcular_valor_total_lista(lista_1)
print(f"Lista: '{lista_1}'\nResultado: {resultado_1}\n")


Testando ferramenta_calcular_valor_total_lista:
Lista: 'teclado, mouse gamer, monitor'
Resultado: O valor total dos itens encontrados é R$ 1249.40.



In [23]:
lista_2 = "headset, cadeira"
resultado_2 = ferramenta_calcular_valor_total_lista(lista_2)
print(f"Lista: '{lista_2}'\nResultado: {resultado_2}\n")


Lista: 'headset, cadeira'
Resultado: O valor total dos itens encontrados é R$ 180.00. Os seguintes itens não foram encontrados e não foram incluídos no cálculo: cadeira.



In [24]:
lista_3 = "mesa, copo"
resultado_3 = ferramenta_calcular_valor_total_lista(lista_3)
print(f"Lista: '{lista_3}'\nResultado: {resultado_3}\n")


Lista: 'mesa, copo'
Resultado: O valor total dos itens encontrados é R$ 0.00. Os seguintes itens não foram encontrados e não foram incluídos no cálculo: mesa, copo.



In [25]:
def run_react_agent(pergunta: str, max_iterations: int = 5) -> str:
    from langchain_ollama import ChatOllama

    model = ChatOllama(model="gemma3:1b", temperature=0)
    mensagens = [("system", PROMPT_REACT), ("human", pergunta)]

    for i in range(max_iterations):
        resposta = str(model.invoke(mensagens).content).strip()
        print(f"\n--- Itera??o {i+1} ---\n{resposta}\n")

        final = re.search(r"Resposta:\s*(.+)", resposta, re.IGNORECASE | re.DOTALL)
        if final:
            return final.group(1).strip()

        acao = re.search(
            r"A??o:\s*([A-Za-z_]+)(?:\s*:\s*([^\n]*))?",
            resposta,
            re.IGNORECASE
        )
        if not acao:
            return f"Erro: nenhuma A??o ou Resposta encontrada. Resposta do modelo: {resposta}"

        nome = acao.group(1).strip().upper()
        argumento = (acao.group(2) or "").strip()

        if nome == "PAUSA":
            return "O modelo respondeu PAUSA sem indicar uma ferramenta."

        ferramentas = {
            "CONSULTAR_ESTOQUE": lambda: consultar_estoque(argumento),
            "CONSULTAR_PRECO_PRODUTO": lambda: consultar_preco_produto(argumento),
            "ENCONTRAR_PRODUTO_MAIS_CARO": ferramenta_encontrar_produto_mais_caro,
            "CALCULAR_VALOR_TOTAL_LISTA": lambda: ferramenta_calcular_valor_total_lista(argumento),
        }
        if nome not in ferramentas:
            return f"Erro: a??o desconhecida: {nome}"

        observacao = ferramentas[nome]()
        print(f"Executou {nome}({argumento!r}): {observacao}")
        mensagens.extend([
            ("assistant", resposta),
            ("human", f"Observa??o: {observacao}\nAgora responda ao usu?rio em uma linha iniciada por Resposta:.")
        ])

    return "Erro: limite de itera??es atingido."


In [26]:
print("Demonstra??o interativa/modelo ignorada na c?lula 25.")


Demonstra??o interativa/modelo ignorada na c?lula 25.


In [27]:
PROMPT_REACT = """
Você funciona em um ciclo de Pensamento, Ação, Pausa e Observação.
Ao final do ciclo, você fornece uma Resposta.
Use "Pensamento" para descrever seu raciocínio.
Use "Ação" para executar ferramentas - e então retorne "PAUSA".
A "Observação" será o resultado da ação executada.
Ações disponíveis:
  - consultar_estoque: retorna a quantidade disponível de um item no inventário (ex: "consultar_estoque: teclado")
  - consultar_preco_produto: retorna o preço unitário de um produto (ex: "consultar_preco_produto: mouse gamer")
  - encontrar_produto_mais_caro: retorna o nome e o preço do produto mais caro no inventário (não requer argumentos)
  - calcular_valor_total_lista: calcula o valor total de uma lista de itens de compra. Recebe uma string com itens separados por vírgula (ex: "teclado, mouse gamer, monitor")

Exemplo:
Pergunta: Quantos monitores temos em estoque?
Pensamento: Devo consultar a ação consultar_estoque para saber a quantidade de monitores.
Ação: consultar_estoque: monitor
PAUSA

Observação: Temos 75 monitores em estoque.
Resposta: Há 75 monitores em estoque.

Exemplo:
Pergunta: Qual é o produto mais caro?
Pensamento: Preciso usar a ação encontrar_produto_mais_caro para descobrir qual produto tem o maior preço.
Ação: encontrar_produto_mais_caro
PAUSA

Observação: O produto mais caro é o(a) monitor com preço de R$ 999.90.
Resposta: O produto mais caro é o(a) monitor com preço de R$ 999.90.

Exemplo:
Pergunta: Quanto custa um teclado e um mouse gamer?
Pensamento: O usuário quer saber o valor total de vários itens. Devo usar a ação calcular_valor_total_lista com os itens "teclado, mouse gamer".
Ação: calcular_valor_total_lista: teclado, mouse gamer
PAUSA

Observação: O valor total dos itens encontrados é R$ 249.50.
Resposta: O valor total do teclado e do mouse gamer é R$ 249.50.
""".strip()


In [28]:
print("Demonstra??o interativa/modelo ignorada na c?lula 27.")


Demonstra??o interativa/modelo ignorada na c?lula 27.


In [29]:
print("Demonstra??o interativa/modelo ignorada na c?lula 28.")


Demonstra??o interativa/modelo ignorada na c?lula 28.
